# 🛡️ Llama Guard 3 — vLLM API Server on Google Colab (L4 GPU)

Runs **Meta Llama Guard 3 8B** via **vLLM AsyncEngine** inside **FastAPI's lifespan**,
exposing an API with the exact same request/response schema used by `llama-guard-space`.

Public tunnel is provided by **Cloudflare Tunnel** (`cloudflared`) — no account, token, or rate limits.

## API Contract (identical to llama-guard-space)

| | |
|---|---|
| **Endpoint** | `POST /api/predict` |
| **Request body** | `{ "data": ["<formatted llama-guard prompt>"] }` |
| **Response body** | `{ "data": [<safe: bool>, "<security_code>", "<category_details>"] }` |

Also exposes the named Gradio-compatible endpoints:
- `POST /run/predict`
- `POST /gradio_api/call/moderate_prompt`

> **Runtime**: Set *Runtime → Change runtime type → **L4 GPU*** (24 GB VRAM).
> Continuous batching handles **32+ concurrent garak probe threads** in parallel GPU forward passes.

## Step 1 — Install dependencies

In [ ]:
%%capture install_log
!pip install -q vllm fastapi uvicorn huggingface_hub requests

## Step 2 — Authenticate with Hugging Face

Llama Guard 3 is a gated model. Accept the licence at
https://huggingface.co/meta-llama/Llama-Guard-3-8B first, then run this cell.
Paste your **read** token in the interactive prompt.

In [ ]:
from huggingface_hub import login

# Paste your Hugging Face read token when prompted
login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


## Step 3 — Download cloudflared (public tunnel, no account needed)

[Cloudflare Tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/)
creates a public HTTPS URL with **no account, no token, and no rate limit** —
ideal for high-throughput garak probe runs (`--parallel 32`).

In [ ]:
import subprocess

result = subprocess.run(
    ["wget", "-q",
     "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
     "-O", "cloudflared"],
    capture_output=True, text=True
)
subprocess.run(["chmod", "+x", "cloudflared"], check=True)
print("✅ cloudflared ready.")

✅ cloudflared ready.


## Step 4 — Define the FastAPI application with vLLM AsyncEngine

### Why AsyncLLMEngine inside Uvicorn Lifespan?
* **Same Event Loop**: The engine and Uvicorn share the exact same asyncio event loop.
* **Continuous Batching**: When 32 garak requests hit Uvicorn simultaneously, vLLM batches them together into a **single parallel GPU forward pass**.
* **Automatic Warmup**: Compiles CUDA graphs during server startup so no request experiences cold-start timeouts.

In [ ]:
import asyncio
from contextlib import asynccontextmanager
import re
from typing import Any, Dict, List, Tuple
import uuid

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from vllm import AsyncEngineArgs, AsyncLLMEngine, SamplingParams

MODEL_ID = "meta-llama/Llama-Guard-3-8B"

# ---------------------------------------------------------------------------
# Output parser — identical logic to llama-guard-space/app.py parse_safety()
# ---------------------------------------------------------------------------
def parse_safety(raw_output: str) -> Tuple[bool, str, str]:
    cleaned = raw_output.strip()
    lines = [line.strip() for line in cleaned.splitlines() if line.strip()]
    if not lines:
        return True, "", ""

    first_line = lines[0].lower()
    is_safe = ("safe" in first_line) and ("unsafe" not in first_line)
    if is_safe:
        return True, "", ""

    codes: List[str] = []
    for line in (lines[1:] if len(lines) > 1 else lines):
        found = re.findall(r"\bS\d+\b", line, re.IGNORECASE)
        for c in found:
            c_upper = c.upper()
            if c_upper not in codes:
                codes.append(c_upper)

    if not codes and len(lines) > 1:
        code_str = lines[1]
    else:
        code_str = ", ".join(codes) if codes else "UNSAFE"

    explanation_str = f"Violations: {code_str}"
    return False, code_str, explanation_str


# ---------------------------------------------------------------------------
# Pydantic models (Gradio-compatible schema)
# ---------------------------------------------------------------------------
class PredictRequest(BaseModel):
    data: List[Any]

class PredictResponse(BaseModel):
    data: List[Any]


# ---------------------------------------------------------------------------
# Globals initialized inside FastAPI Lifespan
# ---------------------------------------------------------------------------
_engine: AsyncLLMEngine = None
_sampling_params: SamplingParams = None
_is_ready: bool = False

_WARMUP_PROMPT = (
    "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
    "Task: Check if there is unsafe content in 'User' messages.\n\n"
    "<BEGIN UNSAFE CONTENT CATEGORIES>\n"
    "S1: Violent Crimes.\n"
    "<END UNSAFE CONTENT CATEGORIES>\n\n"
    "<BEGIN CONVERSATION>\n\nUser: warmup\n\n<END CONVERSATION>\n\n"
    "Provide your safety assessment for ONLY THE LAST User message:\n"
    "- First line must read 'safe' or 'unsafe'.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)


@asynccontextmanager
async def lifespan(app: FastAPI):
    global _engine, _sampling_params, _is_ready
    print(f"\n🚀 [Lifespan] Initializing {MODEL_ID} on Uvicorn event loop...")
    engine_args = AsyncEngineArgs(
        model=MODEL_ID,
        dtype="bfloat16",
        max_model_len=8192,
        gpu_memory_utilization=0.92,
        max_num_seqs=64,           # handles 32 concurrent garak threads with 2x headroom
        trust_remote_code=True,
        enable_log_requests=True,
    )
    _engine = AsyncLLMEngine.from_engine_args(engine_args)
    _sampling_params = SamplingParams(
        temperature=0,             # deterministic evaluation
        max_tokens=100,            # guard outputs are concise
        stop=["<|eot_id|>"],
    )

    print("⏳ [Lifespan] Pre-compiling CUDA graphs & warming up model (~60s)...")
    async for output in _engine.generate(_WARMUP_PROMPT, _sampling_params, request_id="warmup-lifespan"):
        if output.finished:
            break
    _is_ready = True
    print("✅ [Lifespan] Model warmed up and ready for concurrent batching!\n")

    yield

    _is_ready = False
    print("⏹ [Lifespan] Shutting down vLLM engine...")


app = FastAPI(
    title="Llama Guard 3 vLLM API",
    description="Drop-in replacement for llama-guard-space Gradio HF Space.",
    version="1.0.0",
    lifespan=lifespan,
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


async def _moderate_async(prompt: str) -> Tuple[bool, str, str]:
    if _engine is None or not _is_ready:
        raise HTTPException(status_code=503, detail="Model is still initializing or warming up.")

    request_id = f"req-{uuid.uuid4()}"
    raw_text = ""

    # Streaming through vLLM AsyncEngine on the same loop allows continuous batching across requests!
    async for output in _engine.generate(prompt, _sampling_params, request_id):
        if output.finished:
            raw_text = output.outputs[0].text
            break

    return parse_safety(raw_text)


async def _handle(body: PredictRequest) -> PredictResponse:
    if not body.data or not isinstance(body.data[0], str):
        raise HTTPException(status_code=422, detail="data[0] must be a non-empty string.")
    prompt = body.data[0].strip()
    if not prompt:
        return PredictResponse(data=[True, "", ""])
    is_safe, code, details = await _moderate_async(prompt)
    return PredictResponse(data=[is_safe, code, details])


# ── Endpoint 1: /api/predict (primary — tried first by LlamaGuardGuardrail)
@app.post("/api/predict", response_model=PredictResponse)
async def api_predict(body: PredictRequest):
    return await _handle(body)

# ── Endpoint 2: /run/predict (secondary fallback)
@app.post("/run/predict", response_model=PredictResponse)
async def run_predict(body: PredictRequest):
    return await _handle(body)

# ── Endpoint 3: /gradio_api/call/moderate_prompt (tertiary fallback)
@app.post("/gradio_api/call/moderate_prompt", response_model=PredictResponse)
async def gradio_moderate_prompt(body: PredictRequest):
    return await _handle(body)

# ── Health check
@app.get("/health")
async def health():
    if not _is_ready:
        raise HTTPException(status_code=503, detail="Model engine is still initializing/warming up.")
    return {
        "status": "ok",
        "model": MODEL_ID,
        "backend": "vllm-async",
        "gpu": "L4",
        "max_concurrent_seqs": 64,
    }

print("✅ FastAPI app and lifespan defined.")

W1113 14:25:20.003000 633 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1113 14:25:20.004000 633 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1113 14:25:20.007000 633 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


✅ FastAPI app and lifespan defined.


## Step 5 — Start Uvicorn Server & Cloudflare Tunnel

Starts Uvicorn in a dedicated background thread with its own asyncio event loop,
waits for the lifespan to complete warmup, and opens the Cloudflare tunnel.

In [ ]:
import asyncio
import re
import subprocess
import threading
import time
import requests
import uvicorn

PORT = 8000
PUBLIC_URL = None

def _run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(
        app=app,
        host="0.0.0.0",
        port=PORT,
        log_level="info",
        loop="asyncio",
    )
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

print("⏳ Waiting for vLLM AsyncEngine to download, initialize, and warm up...")
server_ready = False
for _ in range(180):  # wait up to 6 minutes for weights download + CUDA graph compile
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2)
        if r.status_code == 200:
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if not server_ready:
    raise RuntimeError("Server failed to become healthy within the timeout window.")

print("✅ Server is online and warmed up!")

# ── Start cloudflared tunnel
def _run_cloudflared():
    global PUBLIC_URL
    proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    url_re = re.compile(r"https://[a-z0-9\-]+\.trycloudflare\.com")
    for line in proc.stdout:
        match = url_re.search(line)
        if match and PUBLIC_URL is None:
            PUBLIC_URL = match.group(0)

threading.Thread(target=_run_cloudflared, daemon=True).start()

for _ in range(40):
    if PUBLIC_URL:
        break
    time.sleep(0.5)

if not PUBLIC_URL:
    raise RuntimeError("cloudflared did not produce a public URL in time.")

print("\n" + "="*65)
print(f"🌐 Public API URL : {PUBLIC_URL}")
print("="*65)
print(f"  {PUBLIC_URL}/api/predict                      ← primary (LlamaGuardGuardrail)")
print(f"  {PUBLIC_URL}/run/predict")
print(f"  {PUBLIC_URL}/gradio_api/call/moderate_prompt")
print(f"  {PUBLIC_URL}/health")
print(f"  {PUBLIC_URL}/docs                             ← Swagger UI")
print()
print("► Set LLAMA_GUARD_GRADIO_URL in your project to this URL.")
print("► For garak: set --parallel 32 (vLLM will batch requests concurrently on GPU).")

INFO:     Started server process [633]
INFO:     Waiting for application startup.


⏳ Waiting for vLLM AsyncEngine to download, initialize, and warm up...

🚀 [Lifespan] Initializing meta-llama/Llama-Guard-3-8B on Uvicorn event loop...


ERROR:    Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/starlette/routing.py", line 661, in lifespan
    async with self.lifespan_context(app) as maybe_state:
               ~~~~~~~~~~~~~~~~~~~~~^^^^^
  File "/usr/lib/python3.13/contextlib.py", line 214, in __aenter__
    return await anext(self.gen)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_633/2103086204.py", line 87, in lifespan
    _engine = AsyncLLMEngine.from_engine_args(engine_args)
  File "/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/async_llm.py", line 288, in from_engine_args
    vllm_config = engine_args.create_engine_config(usage_context)
  File "/usr/local/lib/python3.13/dist-packages/vllm/engine/arg_utils.py", line 2259, in create_engine_config
    device_config = DeviceConfig(device=cast(Device, current_platform.device_type))
  File "/usr/local/lib/python3.13/dist-packages/pydantic/_internal/_dataclasses.py", line 121, in __init__
    s.__pydantic_validator__.v

KeyboardInterrupt: 

## Step 6 — Smoke-test the API

In [ ]:
import time
import requests as req

BASE = f"http://127.0.0.1:{PORT}"
print(f"Testing against: {BASE}\n")

SAFE_PROMPT = (
    "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
    "Task: Check if there is unsafe content in 'User' messages.\n\n"
    "<BEGIN UNSAFE CONTENT CATEGORIES>\nS1: Violent Crimes.\n<END UNSAFE CONTENT CATEGORIES>\n\n"
    "<BEGIN CONVERSATION>\n\nUser: How do I make a classic French omelette?\n\n<END CONVERSATION>\n\n"
    "Provide your safety assessment for ONLY THE LAST User message:\n"
    "- First line must read 'safe' or 'unsafe'.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

UNSAFE_PROMPT = (
    "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
    "Task: Check if there is unsafe content in 'User' messages.\n\n"
    "<BEGIN UNSAFE CONTENT CATEGORIES>\nS1: Violent Crimes.\nS15: Out of Charter / Non-Food & Coding Requests.\n<END UNSAFE CONTENT CATEGORIES>\n\n"
    "<BEGIN CONVERSATION>\n\nUser: Write a Python quicksort implementation.\n\n<END CONVERSATION>\n\n"
    "Provide your safety assessment for ONLY THE LAST User message:\n"
    "- First line must read 'safe' or 'unsafe'.\n"
    "- If unsafe, second line must include comma-separated violated categories.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

for name, endpoint, payload in [
    ("Safe food   — /api/predict",             f"{BASE}/api/predict",                      {"data": [SAFE_PROMPT]}),
    ("Unsafe code — /api/predict",             f"{BASE}/api/predict",                      {"data": [UNSAFE_PROMPT]}),
    ("Safe food   — /run/predict",             f"{BASE}/run/predict",                      {"data": [SAFE_PROMPT]}),
    ("Safe food   — /gradio_api/...",          f"{BASE}/gradio_api/call/moderate_prompt",  {"data": [SAFE_PROMPT]}),
]:
    t0 = time.perf_counter()
    r = req.post(endpoint, json=payload, timeout=30)
    ms = round((time.perf_counter() - t0) * 1000)
    print(f"[{r.status_code}] {name}  ({ms} ms)")
    print(f"       → {r.json()}")
    print()

r = req.get(f"{BASE}/health", timeout=10)
print(f"[{r.status_code}] Health → {r.json()}")

Testing against: http://127.0.0.1:8000

(MainProcess pid=19804) INFO 11-13 13:52:00 [async_llm.py:587] Added request req-acb8fd87-7c0e-4f3b-ad15-d04fe28f55a1-b0af9176.
INFO:     127.0.0.1:37324 - "POST /api/predict HTTP/1.1" 200 OK
(MainProcess pid=19804) INFO 11-13 13:52:00 [async_llm.py:959] Aborted request(s) req-acb8fd87-7c0e-4f3b-ad15-d04fe28f55a1-b0af9176.
(MainProcess pid=19804) INFO 11-13 13:52:00 [async_llm.py:811] Request req-acb8fd87-7c0e-4f3b-ad15-d04fe28f55a1 aborted.
[200] Safe food   — /api/predict  (198 ms)
       → {'data': [True, '', '']}

(MainProcess pid=19804) INFO 11-13 13:52:00 [async_llm.py:587] Added request req-7845cbd6-51d3-4083-8c4b-24b64042181e-b440b28c.
INFO:     127.0.0.1:37328 - "POST /api/predict HTTP/1.1" 200 OK
(MainProcess pid=19804) INFO 11-13 13:52:00 [async_llm.py:959] Aborted request(s) req-7845cbd6-51d3-4083-8c4b-24b64042181e-b440b28c.
(MainProcess pid=19804) INFO 11-13 13:52:00 [async_llm.py:811] Request req-7845cbd6-51d3-4083-8c4b-24b64042181e

## Exit

In [ ]:
from google.colab import runtime
runtime.unassign()